# Dollar Conveyor — Interactive Diagnostic
**Venugopal (2026)** · Settlement-Driven Dollar Demand

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chanakyan/thirdbuyeradvisory.github.io/blob/main/dollar_conveyor_widgets.ipynb)

**Run Cell 1 first (Colab setup), then run all remaining cells. Move sliders. Zone updates instantly.**

In [ ]:
# ── Colab setup -- run this cell first ──────────────────────────────────
import sys
if 'google.colab' in sys.modules:
    !pip install ipywidgets -q
    from google.colab import output
    output.enable_custom_widget_manager()
    print('Colab widget manager enabled.')
else:
    print('Not Colab -- skipping. Run normally.')


# Dollar Conveyor — Interactive Diagnostic
**Venugopal (2026)** · Settlement-Driven Dollar Demand

Run all cells. Move sliders. Zone, cliff distance, and scissors trajectory update instantly.

Bugs fixed: Zone III messaging, crossing detection when already below threshold, double-count guard on drain.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'axes.linewidth': 0.6,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'font.family': 'serif',
    'font.size': 9,
})
INK = '#1a1a1a'; DIM = '#888888'
RED = '#8B1A1A'; RUST = '#A0522D'; SAGE = '#4A6741'

def rangeframe(ax, xs, ys, pad=0.04):
    xp = (np.max(xs) - np.min(xs)) * pad
    yp = (np.max(ys) - np.min(ys)) * pad
    ax.spines['bottom'].set_bounds(np.min(xs) - xp, np.max(xs) + xp)
    ax.spines['left'].set_bounds(max(0, np.min(ys) - yp), np.max(ys) + yp)
print('Setup complete.')


## Model — equations 9, 31, Appendix B

In [ ]:
def sddd_floor(E, tau, k, delta, sigma):
    '''Eq. 9: SDDD* = (E*tau/k)*(1 + delta*sigma)'''
    return (E * tau / k) * (1 + delta * sigma)


def compute(R, E, tau, k, F, C, P, SR, L, U,
            sigma, R_trend_non_tariff, export_USD, tariff_hit, T_IMF):
    '''
    Bug fix 3: R_trend_non_tariff is the reserve change EXCLUDING tariff effects.
    Export loss from tariff is computed separately and added.
    Total drain = |R_trend_non_tariff| + export_loss + floor_rise.
    No double-counting.
    '''
    # delta (eq. 8, equal weights)
    delta = np.mean([F, C, P, SR, L, 1 - U])

    # SDDD floor (eq. 9)
    SDDD = sddd_floor(E, tau, k, delta, sigma)

    # financing gap compression Psi (Appendix C)
    # integral of floor over T_IMF months -- floor rises as sigma drifts
    sigma_drift = 0.02          # per year under sustained shock
    t_imf_months = np.linspace(0, T_IMF, 60)
    floors_during_gap = sddd_floor(E, tau, k, delta,
                                   sigma + sigma_drift * (t_imf_months / 12))
    # Psi = integral of floor over gap / 12 (convert months to years)
    _integrate = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
    Psi = _integrate(floors_during_gap, t_imf_months / 12)

    # hysteresis penalty H = 0 (kappa=0 assumed; journalist can note impairment)
    Theta = SDDD + Psi

    # zone
    if R < SDDD:
        zone = 'III'
    elif R < Theta:
        zone = 'II'
    else:
        zone = 'I'

    # drain components (no double-counting)
    export_loss   = export_USD * tariff_hit           # USD bn/yr
    floor_rise    = sddd_floor(E, tau, k, delta,
                               sigma + sigma_drift) - SDDD  # USD bn/yr
    net_drain     = abs(min(R_trend_non_tariff, 0)) + export_loss + floor_rise

    # cliff and Theta buffers
    cliff_buffer = R - SDDD
    theta_buffer = R - Theta

    # Bug fix 1+2: explicit handling for all zone states
    if cliff_buffer <= 0:
        months_to_cliff = 0        # already below cliff
        months_to_theta = 0
    elif theta_buffer <= 0:
        months_to_theta = 0        # already below Theta*
        months_to_cliff = (cliff_buffer / net_drain * 12) if net_drain > 0 else np.inf
    else:
        months_to_theta = (theta_buffer / net_drain * 12) if net_drain > 0 else np.inf
        months_to_cliff = (cliff_buffer  / net_drain * 12) if net_drain > 0 else np.inf

    # 10-year trajectory
    years     = np.linspace(0, 10, 500)
    R_path    = np.maximum(R + R_trend_non_tariff * years - export_loss * years, 0)
    floor_path = sddd_floor(E, tau, k, delta, sigma + sigma_drift * years)
    theta_path = floor_path + Psi

    # Bug fix 2: crossing detection handles already-below case
    cross_theta, cross_cliff = None, None
    if R <= Theta:
        cross_theta = (0, R)           # already crossed before chart starts
    else:
        below = np.where(R_path < theta_path)[0]
        if len(below):
            cross_theta = (years[below[0]], R_path[below[0]])

    if R <= SDDD:
        cross_cliff = (0, R)           # already crossed
    else:
        below = np.where(R_path < floor_path)[0]
        if len(below):
            cross_cliff = (years[below[0]], R_path[below[0]])

    return dict(
        delta=delta, SDDD=SDDD, Theta=Theta, zone=zone,
        cliff_buffer=cliff_buffer, theta_buffer=theta_buffer,
        net_drain=net_drain, export_loss=export_loss, floor_rise=floor_rise,
        months_to_theta=months_to_theta, months_to_cliff=months_to_cliff,
        years=years, R_path=R_path, floor_path=floor_path, theta_path=theta_path,
        cross_theta=cross_theta, cross_cliff=cross_cliff,
        R_trend=R_trend_non_tariff,
    )

print('Model loaded.')


## Plot function

In [ ]:
def plot_diagnostic(country, d, R):
    zone_col = {1: SAGE, 2: RUST, 3: RED}[int(d['zone'])]
    zone_text = {
        '1': 'Stressed — self-recoverable',
        '2': 'External intervention necessary',
        '3': 'PIPELINE FAILURE — import collapse categorical',
    }[d['zone']]

    fig = plt.figure(figsize=(14, 7))
    fig.patch.set_facecolor('white')
    gs = gridspec.GridSpec(2, 3, figure=fig,
                           hspace=0.55, wspace=0.38,
                           height_ratios=[1, 1.8])

    # ── PANEL 1: Zone gauge (number line) ────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    x_max = max(R * 1.1, d['Theta'] * 1.2, 2.0)

    ax1.axhline(0, color='#cccccc', lw=0.8, xmin=0, xmax=1)

    # SDDD* tick
    ax1.plot([d['SDDD'], d['SDDD']], [-0.4, 0.4], color=RED, lw=1.4)
    ax1.text(d['SDDD'], 0.55, f"SDDD*\n${d['SDDD']:.2f}B",
             color=RED, fontsize=8, ha='center')

    # Theta* tick
    ax1.plot([d['Theta'], d['Theta']], [-0.4, 0.4], color=RUST, lw=1.4, ls='--')
    ax1.text(d['Theta'], -0.70, f"\u0398*\n${d['Theta']:.2f}B",
             color=RUST, fontsize=8, ha='center')

    # country dot
    ax1.scatter(R, 0, color=zone_col, s=120, zorder=6)
    ax1.text(R, 0.55, f"{country}\n${R:.1f}B",
             color=zone_col, fontsize=9, ha='center', fontweight='bold')

    ax1.set_xlim(0, x_max)
    ax1.set_ylim(-1, 1)
    ax1.set_yticks([])
    ax1.spines['left'].set_visible(False)
    ax1.spines['bottom'].set_bounds(0, x_max)
    ax1.set_xlabel('USD billion')
    ax1.set_title(
        f"Zone {d['zone']} — {zone_text}",
        color=zone_col, fontsize=11, pad=6
    )

    # ── PANEL 2: SDDD floor vs delta ─────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    dv = np.linspace(0, 1, 200)
    f_lo  = (1.3 * 3 / 1.5) * (1 + dv * 0.15)
    f_now = (1.3 * 3 / 1.5) * (1 + dv * d['SDDD'] /
             ((1.3 * 3 / 1.5) * (1 + d['delta'] * 0.01 + 0.01)))
    # recompute cleanly from actual params
    E_ref, tau_ref, k_ref = 1.3, 3, 1.5
    f_curr = (E_ref * tau_ref / k_ref) * (1 + dv * 0.30)
    f_hi   = (E_ref * tau_ref / k_ref) * (1 + dv * 0.50)

    ax2.plot(dv, f_lo,   color=DIM, lw=0.9, ls='--')
    ax2.plot(dv, f_curr, color=INK, lw=1.4)
    ax2.plot(dv, f_hi,   color=RED, lw=0.9, ls='--')

    ax2.scatter(d['delta'], d['SDDD'], color=RED, s=40, zorder=5)
    ax2.plot([d['delta'], d['delta']], [0, d['SDDD']],
             color=RED, lw=0.5, ls=':')
    ax2.text(d['delta'] + 0.03, d['SDDD'] + 0.05,
             f"{country}\n\u03b4={d['delta']:.2f}\n${d['SDDD']:.2f}B",
             color=RED, fontsize=7.5)

    ax2.text(0.62, f_lo[-1] * 0.94, '\u03c3=0.15', color=DIM, fontsize=7)
    ax2.text(0.62, f_hi[-1] * 0.97, '\u03c3=0.50', color=RED, fontsize=7)

    ax2.set_xlabel('\u03b4 (domestic sensitivity)')
    ax2.set_ylabel('SDDD floor (USD billion)')
    ax2.set_title('Floor — US rates not in this equation', pad=6)
    rangeframe(ax2, dv, f_hi)
    ax2.set_xticks([0, 0.25, 0.5, 0.75, 1.0])

    # ── PANEL 3: Scissors trajectory ─────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1:])
    yrs   = d['years']
    Rp    = d['R_path']
    fp    = d['floor_path']
    tp    = d['theta_path']

    ax3.plot(yrs, fp, color=RED,  lw=0.8, ls=':',  zorder=2)
    ax3.plot(yrs, tp, color=RUST, lw=0.8, ls='--', zorder=2)
    ax3.plot(yrs, Rp, color=INK,  lw=1.8, zorder=3)

    # end labels
    ax3.text(yrs[-1] + 0.1, Rp[-1],  country,  color=INK,  fontsize=8, va='center')
    ax3.text(yrs[-1] + 0.1, fp[-1],  'SDDD*',  color=RED,  fontsize=7, va='center')
    ax3.text(yrs[-1] + 0.1, tp[-1],  '\u0398*', color=RUST, fontsize=7, va='center')

    # crossing dots -- Bug fix 2: works for already-below case
    if d['cross_theta']:
        tx, ty = d['cross_theta']
        ax3.scatter(tx, ty, color=RUST, s=30, zorder=6)
        label = 'already below \u0398*' if tx == 0 else f'\u0398* yr {tx:.1f}'
        ax3.text(tx + 0.1, ty + 0.3, label, color=RUST, fontsize=7.5)

    if d['cross_cliff']:
        cx, cy = d['cross_cliff']
        ax3.scatter(cx, cy, color=RED, s=30, zorder=6)
        label = 'already below cliff' if cx == 0 else f'cliff yr {cx:.1f}'
        ax3.text(cx + 0.1, cy + 0.3, label, color=RED, fontsize=7.5)

    # drain breakdown annotation
    ax3.text(0.15, max(Rp) * 0.12,
             f"drain: ${d['net_drain']:.1f}B/yr\n"
             f"  trend: ${min(d['R_trend'],0):.1f}B/yr\n"
             f"  tariff: ${-d['export_loss']:.1f}B/yr\n"
             f"  floor rise: ${-d['floor_rise']:.2f}B/yr",
             color=DIM, fontsize=7.5)

    ax3.set_xlabel('Years from now')
    ax3.set_ylabel('USD billion')
    ax3.set_title('Scissors trajectory — 10-year horizon at current burn rate', pad=6)
    all_y = np.concatenate([Rp, fp, tp])
    rangeframe(ax3, yrs, all_y[all_y > 0])
    ax3.set_xlim(0, 11.5)

    # Bug fix 1: Zone III explicit messaging
    if d['zone'] == 'III':
        ax3.text(0.5, (max(Rp) + max(tp)) / 2,
                 'ALREADY BELOW SDDD*\nPIPELINE FAILURE IN PROGRESS',
                 color=RED, fontsize=13, fontweight='bold', alpha=0.25,
                 ha='center', rotation=10)

    # suptitle: timeline
    if d['zone'] == 'III':
        timeline = f"{country} is below the cliff. Pipeline failure is categorical."
    elif d['zone'] == 'II':
        mc = d['months_to_cliff']
        mc_str = f"{mc:.0f} months" if np.isfinite(mc) else "not at current burn rate"
        timeline = (f"{country} is below \u0398*. External intervention necessary. "
                    f"Cliff crossing in {mc_str}.")
    else:
        mt = d['months_to_theta']
        mc = d['months_to_cliff']
        mt_str = f"{mt:.0f} mo" if np.isfinite(mt) else "\u221e"
        mc_str = f"{mc:.0f} mo" if np.isfinite(mc) else "\u221e"
        timeline = (f"{country} Zone I. "
                    f"\u0398* in {mt_str} · cliff in {mc_str} at current burn rate. "
                    f"SDDD*=${d['SDDD']:.2f}B · \u0398*=${d['Theta']:.2f}B · "
                    f"\u03b4={d['delta']:.2f}")

    fig.suptitle(timeline, fontsize=9, color=zone_col, y=1.01)
    plt.tight_layout()
    plt.show()

print('Plot function loaded.')


## Interactive diagnostic
Move any slider. Figures and diagnostics update instantly.

In [ ]:
# ── Slider definitions ───────────────────────────────────────────────────
style   = {'description_width': '220px'}
layout  = widgets.Layout(width='520px')

w_country = widgets.Text(value='Bangladesh', description='Country',
                         style=style, layout=layout)

# Reserves & imports
w_R       = widgets.FloatSlider(value=20.0, min=0.1,  max=60,  step=0.5,
    description='R: usable reserves ($B)', style=style, layout=layout)
w_E       = widgets.FloatSlider(value=1.3,  min=0.1,  max=6.0, step=0.1,
    description='E: essential imports ($B/mo)', style=style, layout=layout)
w_trend   = widgets.FloatSlider(value=-1.0, min=-12,  max=5,   step=0.5,
    description='Reserve trend excl. tariff ($B/yr)', style=style, layout=layout)
w_tau     = widgets.IntSlider(value=3, min=1, max=6,
    description='tau: LC pipeline tenor (months)', style=style, layout=layout)

# Volatility & shock
w_sigma   = widgets.FloatSlider(value=0.30, min=0.05, max=0.70, step=0.01,
    description='\u03c3: commodity volatility', style=style, layout=layout)
w_export  = widgets.FloatSlider(value=38.0, min=1,    max=150,  step=1.0,
    description='Export earnings ($B/yr)', style=style, layout=layout)
w_tariff  = widgets.FloatSlider(value=0.10, min=0,    max=0.50, step=0.01,
    description='Tariff / trade shock (fraction)', style=style, layout=layout)
w_TIMF    = widgets.IntSlider(value=6,    min=1,    max=24,
    description='T_IMF: financing horizon (mo)', style=style, layout=layout)

# Sensitivity components
w_F  = widgets.FloatSlider(value=0.50, min=0, max=1, step=0.05,
    description='F: fiscal exposure (1=frozen price)', style=style, layout=layout)
w_C  = widgets.FloatSlider(value=0.70, min=0, max=1, step=0.05,
    description='C: consumption centrality', style=style, layout=layout)
w_P  = widgets.FloatSlider(value=0.70, min=0, max=1, step=0.05,
    description='P: political pass-through', style=style, layout=layout)
w_SR = widgets.FloatSlider(value=1.00, min=0, max=1, step=1.00,
    description='SR: strategic reserve (binary)', style=style, layout=layout)
w_L  = widgets.FloatSlider(value=0.60, min=0, max=1, step=0.05,
    description='L: organised protection', style=style, layout=layout)
w_U  = widgets.FloatSlider(value=0.10, min=0, max=1, step=0.05,
    description='U: substitutability (0=none, worse)', style=style, layout=layout)

# ── Layout ───────────────────────────────────────────────────────────────
header = widgets.HTML('<b style="font-family:Georgia">Reserves &amp; Imports</b>')
header2 = widgets.HTML('<b style="font-family:Georgia">Volatility &amp; Shock</b>')
header3 = widgets.HTML('<b style="font-family:Georgia">Sensitivity \u03b4 components (eq. 8)</b>')

controls = widgets.VBox([
    w_country,
    header,  w_R, w_E, w_trend, w_tau,
    header2, w_sigma, w_export, w_tariff, w_TIMF,
    header3, w_F, w_C, w_P, w_SR, w_L, w_U,
])

out = widgets.Output()

def update(_):
    with out:
        clear_output(wait=True)
        d = compute(
            R=w_R.value, E=w_E.value, tau=w_tau.value, k=1.5,
            F=w_F.value, C=w_C.value, P=w_P.value,
            SR=w_SR.value, L=w_L.value, U=w_U.value,
            sigma=w_sigma.value,
            R_trend_non_tariff=w_trend.value,
            export_USD=w_export.value,
            tariff_hit=w_tariff.value,
            T_IMF=w_TIMF.value,
        )
        # printed diagnostic
        zone_sym = {'I': '\u2705', 'II': '\u26a0\ufe0f', 'III': '\U0001f6a8'}[d['zone']]
        print(f"{'='*55}")
        print(f" {zone_sym}  {w_country.value}  |  Zone {d['zone']}")
        print(f"{'='*55}")
        print(f"  \u03b4 (sensitivity)       = {d['delta']:.3f}")
        print(f"  SDDD* floor            = ${d['SDDD']:.3f}B")
        print(f"  \u0398* threshold          = ${d['Theta']:.3f}B")
        print(f"  Cliff buffer           = ${d['cliff_buffer']:.3f}B")
        print(f"  \u0398* buffer             = ${d['theta_buffer']:.3f}B")
        print()
        if d['zone'] == 'III':
            print(f"  *** BELOW CLIFF. Pipeline failure is categorical. ***")
        elif d['zone'] == 'II':
            mc = d['months_to_cliff']
            mc_s = f"{mc:.0f} months" if np.isfinite(mc) else "not at current burn"
            print(f"  Already below \u0398*.  Cliff in {mc_s}.")
            print(f"  External intervention structurally necessary.")
        else:
            mt = d['months_to_theta']
            mc = d['months_to_cliff']
            mt_s = f"{mt:.0f} mo" if np.isfinite(mt) else "\u221e"
            mc_s = f"{mc:.0f} mo" if np.isfinite(mc) else "\u221e"
            print(f"  \u0398* entry:  {mt_s}  |  Cliff: {mc_s}")
        print()
        print(f"  Net drain              = ${d['net_drain']:.2f}B/yr")
        print(f"    reserve trend        : ${min(d['R_trend'],0):.2f}B/yr")
        print(f"    export loss (tariff) : ${-d['export_loss']:.2f}B/yr")
        print(f"    SDDD floor rise      : ${-d['floor_rise']:.3f}B/yr")
        print()
        print(f"  Corollary B.2: the tariff changes export composition.")
        print(f"  It does not reduce the LC requirement by one dollar.")
        print(f"{'='*55}")
        plot_diagnostic(w_country.value, d, w_R.value)

# wire all sliders to update
for w in [w_country, w_R, w_E, w_trend, w_tau, w_sigma, w_export,
          w_tariff, w_TIMF, w_F, w_C, w_P, w_SR, w_L, w_U]:
    w.observe(update, names='value')

display(widgets.HBox([controls, out]))
update(None)  # render on load


## Presets — 2022 crisis countries
Run a cell to load that country's calibrated parameters into the sliders above.

In [ ]:
def load_preset(name, R, E, tau, sigma, R_trend, export_USD, tariff_hit,
                F, C, P, SR, L, U, T_IMF=6):
    w_country.value = name
    w_R.value = R; w_E.value = E; w_tau.value = tau; w_sigma.value = sigma
    w_trend.value = R_trend; w_export.value = export_USD
    w_tariff.value = tariff_hit; w_TIMF.value = T_IMF
    w_F.value = F; w_C.value = C; w_P.value = P
    w_SR.value = SR; w_L.value = L; w_U.value = U

presets = {
    'Bangladesh 2025': dict(name='Bangladesh', R=20.0, E=1.30, tau=3, sigma=0.30,
        R_trend=-1.0, export_USD=38.0, tariff_hit=0.10,
        F=0.50, C=0.70, P=0.70, SR=1.0, L=0.60, U=0.10, T_IMF=6),
    'Pakistan Feb-23': dict(name='Pakistan', R=2.9,  E=1.30, tau=2, sigma=0.35,
        R_trend=-2.0, export_USD=27.0, tariff_hit=0.08,
        F=0.55, C=0.65, P=0.65, SR=1.0, L=0.55, U=0.10, T_IMF=6),
    'Sri Lanka Apr-22': dict(name='Sri Lanka', R=0.05, E=0.30, tau=2, sigma=0.40,
        R_trend=-3.0, export_USD=12.0, tariff_hit=0.05,
        F=0.60, C=0.70, P=0.75, SR=1.0, L=0.50, U=0.05, T_IMF=11),
    'Egypt 2024': dict(name='Egypt', R=35.0, E=2.50, tau=3, sigma=0.30,
        R_trend=-3.0, export_USD=40.0, tariff_hit=0.07,
        F=0.96, C=0.75, P=0.80, SR=1.0, L=0.70, U=0.05, T_IMF=18),
}

# ── Run whichever line you need ──────────────────────────────────────────
load_preset(**presets['Bangladesh 2025'])
# load_preset(**presets['Pakistan Feb-23'])
# load_preset(**presets['Sri Lanka Apr-22'])
# load_preset(**presets['Egypt 2024'])
